**Open Reading Frames (ORFs) in DNA sequences.**
DNA is read in triplets called **codons**, each encoding one amino acid.
An **Open Reading Frame** is a stretch of DNA that begins with a start codon
(`ATG`, which encodes Methionine) and ends with a stop codon
(`TAA`, `TAG`, or `TGA`).
The sequence of codons between start and stop encodes a protein.

Because codons are three bases long, any DNA sequence has six possible
reading frames: three on the forward strand (offsets 0, 1, 2) and three
on the reverse complement strand (offsets 0, 1, 2).
This notebook builds a pipeline to find all valid ORFs across all six frames.

In [ ]:
"""seq_orf.ipynb"""

# Cell 01 - Genetic code: mapping from amino acid to its codons
# Keys are (3-letter code, 1-letter symbol) tuples.
# Values are lists of all codons (triplets) that encode that amino acid.

from pprint import pprint

AMINO_ACIDS = {
    ("Ala", "A"): ["GCT", "GCA", "GCC", "GCG"],  # Alanine
    ("Arg", "R"): ["CGT", "CGC", "CGA", "CGG", "AGA", "AGG"],  # Arginine
    ("Asn", "N"): ["AAT", "AAC"],  # Asparagine
    ("Asp", "D"): ["GAT", "GAC"],  # Aspartic Acid
    ("Cys", "C"): ["TGT", "TGC"],  # Cysteine
    ("Gln", "Q"): ["CAA", "CAG"],  # Glutamine
    ("Glu", "E"): ["GAA", "GAG"],  # Glutamic Acid
    ("Gly", "G"): ["GGT", "GGC", "GGA", "GGG"],  # Glycine
    ("His", "H"): ["CAT", "CAC"],  # Histidine
    ("Ile", "I"): ["ATT", "ATC", "ATA"],  # Isoleucine
    ("Leu", "L"): ["TTA", "TTG", "CTT", "CTC", "CTA", "CTG"],  # Leucine
    ("Lys", "K"): ["AAA", "AAG"],  # Lysine
    ("Met", "M"): ["ATG"],  # Methionine (Start)
    ("Phe", "F"): ["TTT", "TTC"],  # Phenylalanine
    ("Pro", "P"): ["CCT", "CCC", "CCA", "CCG"],  # Proline
    ("Ser", "S"): ["TCT", "TCC", "TCA", "TCG", "AGT", "AGC"],  # Serine
    ("Thr", "T"): ["ACT", "ACC", "ACA", "ACG"],  # Threonine
    ("Trp", "W"): ["TGG"],  # Tryptophan
    ("Tyr", "Y"): ["TAT", "TAC"],  # Tyrosine
    ("Val", "V"): ["GTT", "GTC", "GTA", "GTG"],  # Valine
    ("Stop", "0"): ["TAA", "TGA", "TAG"],  # Stop codons
}

# Build the inverted lookup once at module level: codon -> (3-letter, 1-letter)
# This avoids rebuilding it on every call to decode_codons()
_CODON_TO_ACID = {codon: key for key, codons in AMINO_ACIDS.items() for codon in codons}

pprint(AMINO_ACIDS)

**The reverse complement.**
DNA is double-stranded: each base pairs with its complement
($A \leftrightarrow T$, $C \leftrightarrow G$).
The second strand runs in the opposite direction (antiparallel), so
the reverse complement is the sequence read backwards with each base
replaced by its complement.
ORFs can occur on either strand, so all six reading frames must be checked.

`str.translate` with a translation table built by `str.maketrans` is the
idiomatic Python way to do character-by-character substitution - it is both
cleaner and faster than a character-by-character loop.

In [ ]:
# Cell 02 - Reverse complement of a DNA sequence

_COMPLEMENT = str.maketrans("ACGT", "TGCA")


def reverse_complement(seq: str) -> str:
    """Return the reverse complement of a DNA sequence."""
    return seq[::-1].translate(_COMPLEMENT)


print(f"reverse_complement('ATCG') = {reverse_complement('ATCG')}")
# A<->T and C<->G: ATCG reversed is GCTA, complement is CGAT
print(
    f"reverse_complement('AACAAGTTTACAAGC') = {reverse_complement('AACAAGTTTACAAGC')}"
)

**Reading frames and codons.**
A reading frame starts at a given `offset` (0, 1, or 2) and reads
non-overlapping triplets from that position onward.
Any trailing bases that do not form a complete triplet are ignored.
For example, `"ATCGGAT"` at offset 0 gives `["ATC", "GGA"]`;
the final `"T"` is discarded.

In [ ]:
# Cell 03 - Split a sequence into codons starting at a given offset
# range stop is len(seq)-2 so that i+3 <= len(seq) is always satisfied


def make_codons(seq: str, offset: int) -> list[str]:
    """Return a list of non-overlapping triplets starting at offset."""
    return [seq[i : i + 3] for i in range(offset, len(seq) - 2, 3)]


print(f"offset 0: {make_codons('ATCGGAT', 0)}")  # ['ATC', 'GGA']
print(f"offset 1: {make_codons('ATCGGAT', 1)}")  # ['TCG', 'GAT']
print(f"offset 2: {make_codons('ATCGGAT', 2)}")  # ['CGG'] - only one full triplet

**Finding a codon by position.**
`list.index` returns the position of the first match but raises `ValueError`
if the codon is absent.
Wrapping it in a `try/except` and returning `-1` on failure gives a
convenient sentinel value that the ORF finder uses to detect missing
start or stop codons.

In [ ]:
# Cell 04 - Find the index of a codon within a codon list (-1 if absent)


def find_codon(codon_list: list[str], codon: str) -> int:
    """Return the index of codon in codon_list, or -1 if not present."""
    try:
        return codon_list.index(codon)
    except ValueError:
        return -1


# offset 0: ['ATC', 'GGA'] - GGA is at index 1
print(f"offset 0, find GGA: {find_codon(make_codons('ATCGGAT', 0), 'GGA')}")
# offset 1: ['TCG', 'GAT'] - TCG is at index 0
print(f"offset 1, find TCG: {find_codon(make_codons('ATCGGAT', 1), 'TCG')}")
# offset 2: ['CGG'] - GAT is not present (it straddles a codon boundary)
print(f"offset 2, find GAT: {find_codon(make_codons('ATCGGAT', 2), 'GAT')}")

**Decoding codons to amino acid symbols.**
Each codon maps to an amino acid via `_CODON_TO_ACID`, the inverted
lookup built once in Cell 01.
The 1-letter symbol is element `[1]` of the key tuple.
The trailing stop codon symbol is stripped before returning since stop
codons terminate translation but do not encode an amino acid.

In [ ]:
# Cell 05 - Translate a space-separated codon string to 1-letter amino acid symbols


def decode_codons(codon_string: str) -> str:
    """Return the 1-letter amino acid sequence for a space-separated codon string.

    The trailing stop codon is consumed but not included in the output.
    """
    symbols = [
        _CODON_TO_ACID[codon][1]  # [1] selects the 1-letter symbol from the tuple key
        for codon in codon_string.split()
    ]
    return "".join(symbols[:-1])  # drop the stop codon symbol


# ATG->M, GGA->G, GTC->V, GAT->D, TAG->stop (dropped)
print(decode_codons("ATG GGA GTC GAT TAG"))

**Finding an Open Reading Frame.**
An ORF requires:
1. A start codon `ATG` somewhere in the codon list.
2. A stop codon (`TAA`, `TAG`, or `TGA`) after the start codon.
3. At least one coding codon between start and stop.

When multiple stop codons appear after the start, the nearest one is used
since the ribosome stops at the first stop codon it encounters.

In [ ]:
# Cell 06 - Extract an ORF from a sequence at a given reading frame offset


def get_orf(seq: str, offset: int) -> str | None:
    """Return a string of codons for the first valid ORF at the given offset,
    or None if no valid ORF exists.
    """
    codon_list = make_codons(seq, offset)

    # Find the start codon
    start_idx = find_codon(codon_list, "ATG")
    if start_idx < 0:
        return None

    # Find all three possible stop codons; keep only those after the start
    stop_indexes = [
        idx
        for idx in [
            find_codon(codon_list, "TAA"),
            find_codon(codon_list, "TAG"),
            find_codon(codon_list, "TGA"),
        ]
        if idx > start_idx
    ]
    if not stop_indexes:
        return None

    stop_idx = min(stop_indexes)  # first stop codon after start

    if stop_idx == start_idx + 1:  # no coding codons between start and stop
        return None

    # Build the codon string from start to stop (inclusive)
    orf_codons = " ".join(codon_list[start_idx : stop_idx + 1])

    return f"{orf_codons}  ({decode_codons(orf_codons)})"


print(get_orf("ATGGGAGTCGATTAG", 0))  # ATG GGA GTC GAT TAG -> MGVD
print(get_orf("ATGGGAGTCGATTAG", 1))  # no valid ORF at offset 1

**Searching all six reading frames.**
A complete ORF search scans the forward strand at offsets 0, 1, 2
and then the reverse complement strand at offsets 0, 1, 2.
Genes can be encoded on either strand of DNA, so all six frames
must be checked.

In [ ]:
# Cell 07 - Read a DNA sequence file and report all ORFs across six reading frames

import re
from pathlib import Path

_NON_LETTER = re.compile("[^A-Z]")


def analyze_file(file_name: str) -> None:
    """Read a DNA file, clean it, and print all ORFs found in six reading frames."""
    seq = Path(file_name).read_bytes().decode().upper()
    seq = _NON_LETTER.sub("", seq)

    seq_rc = reverse_complement(seq)

    print(f"Forward sequence    : {seq}")
    for offset in range(3):
        if orf := get_orf(seq, offset):
            print(f"  Forward frame +{offset}  : {orf}")

    print(f"Reverse complement  : {seq_rc}")
    for offset in range(3):
        if orf := get_orf(seq_rc, offset):
            print(f"  Reverse frame +{offset}  : {orf}")


analyze_file("dna_sequence.txt")